# 教程 2：写自己的训练头 —— 从一行 loss 到 CNN

**场景**（ESamp 类）：模型每生成一个 token，就用"源层 hidden → 目标层 hidden"
的蒸馏信号现场训练一个小头，并用小头的预测引导后续采样。

观测（教程 1）只读状态；这一篇引入**会变的参数**：

1. 最小的可训练 workload：一个线性头 + 一行 loss；
2. 让训练影响生成：采样引导与"还没训练好别说话"门控；
3. 把头换成 CNN：骨架一行不动。

新概念只有一个：**带优化器的状态**。`RequestState(..., optimizer=pllm.AdamW(...))`
声明的字段由运行时负责版本化、按请求隔离、逐步发布 —— 你的 `loss` 函数只返回
一个标量（或每条 lineage 一个标量），梯度与更新由框架按签名完成。

## 第一步：最小的可训练 workload

逐行读这个类：

- `head` 是**请求状态**：每个请求一份 `w`（`(H, H)` 矩阵），AdamW 训练；
- `loss` 上的 `@pllm.train(state="head", ...)` 声明：这个函数的返回值是损失，
  梯度流向 `head` 的字段，更新走发布纪律；
- `Annotated[torch.Tensor, pllm.Hidden(0)]` / `Hidden(-1)` 取源层/目标层 hidden；
- `groups`（`StateIndex`）告诉你每行属于哪条 lineage —— 损失按 lineage 归并，
  多请求同批训练互不串味；
- `predict` + `@pllm.guide_sampling(beta=...)` 是**效果**：把预测经原采样器投影进
  logits。没有效果的纯训练 workload 不构成后端附件 —— 训练必须有个"用处"。

In [1]:
from typing import Annotated
import torch
import pllm
from vllm import SamplingParams
from pllm.config import VllmDType, ModelConfig


class LinearDistiller(pllm.Workload):
    """每步从第 0 层 hidden 学一个线性映射到最后一层 hidden。"""

    head = pllm.RequestState(
        {"w": lambda m: (m.hidden_size, m.hidden_size)},
        dtype=torch.float32,
        optimizer=pllm.AdamW(lr=3e-3),
        keys=(pllm.RequestKeyField.PROMPT_IDX, pllm.RequestKeyField.SAMPLE_IDX),
    )

    def ready(self, versions: Annotated[torch.Tensor, pllm.StateVersion("head")]) -> torch.Tensor:
        return versions > 0     # 首个版本发布前不引导，避免零初始化参数注入噪声

    @pllm.on(pllm.Phase.BEFORE_SAMPLE, rows="ready")
    @pllm.guide_sampling(beta=0.2)
    def predict(self, source: Annotated[torch.Tensor, pllm.Hidden(0)],
                w: Annotated[torch.Tensor, pllm.Parameter("head", "w")],
                versions: Annotated[torch.Tensor, pllm.StateVersion("head")]) -> torch.Tensor:
        return torch.bmm(source.float().unsqueeze(1), w).squeeze(1)

    @pllm.on(pllm.Phase.AFTER_SAMPLE)
    @pllm.train(state="head", reduction="state_mean")
    def loss(self, source: Annotated[torch.Tensor, pllm.Hidden(0)],
             target: Annotated[torch.Tensor, pllm.Hidden(-1)],
             w: Annotated[torch.Tensor, pllm.Parameter("head", "w")],
             groups: Annotated[torch.Tensor, pllm.StateIndex("head")]) -> torch.Tensor:
        prediction = torch.bmm(source.float().unsqueeze(1), w).squeeze(1)
        losses = (prediction - target.float()).square().mean(1)   # 每行一个损失
        counts = torch.bincount(groups).to(losses)
        return losses.new_zeros(counts.shape).index_add(0, groups, losses) / counts


llm = pllm.make_llm(workload=LinearDistiller(), config=ModelConfig(
    "Qwen/Qwen2.5-0.5B-Instruct", VllmDType.BFLOAT16, 64, 2, 128, 0.2, seed=7))
outputs = llm.generate(["The sky is", "Water freezes at"],
    SamplingParams(max_tokens=8, temperature=0.0, ignore_eos=True), use_tqdm=False)
d = llm.worker_state().diagnostics
print("losses:", d.loss_count, "updates:", d.update_count, "fallback:", d.fallback_count)
for output in outputs:
    print(output.outputs[0].text)

/workspace/pllm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-10 13:13:42,617	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 09-10 13:13:43 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 64, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 128, 'max_num_seqs': 2, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 13:13:44 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 13:13:45 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 13:13:45 [model.py:1883] Using max model len 64


... (27 outputs left out)

INFO 09-10 13:13:53 [core.py:355] init engine (profile, create kv cache, warmup model) took 1.60 s


INFO 09-10 13:13:56 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


WARNING 09-10 13:13:56 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _bmm_outer_product_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


losses: 14 updates: 14 fallback: 0
 a beautiful place. It is a place
 0 degrees Celsius and boils at 


## 第二步：看见训练 —— 逐步损失

训练任务的损失在选择器开启时会进 trace。打开 `TraceSelection(tasks=..., tensors=True)`，
每个训练步都会留下一条 `loss` 事件。task id 从签名计划里按符号名查。

In [2]:
from pllm.execution.trace import TraceSelection

plan = llm.attachment.inputs.plan
index = next(i for i, task in enumerate(plan.semantic_graph.tasks)
             if str(task.symbol) == "loss")
loss_id = next(task.task_id for task in plan.schedule_physical.tasks
               if task.task_id == index)

llm.observe(TraceSelection(tasks=(loss_id,), tensors=True, max_tensor_bytes=32768))
llm.generate(["The sea is"], SamplingParams(max_tokens=6, temperature=0.0,
    ignore_eos=True), use_tqdm=False)
llm.observe(TraceSelection())   # 关掉选择器

for event in llm.trace_snapshot().events:
    if event.tensor_name == "loss":
        print(f"step {event.step_id}: loss={event.tensor().item():.4f}")

step 9: loss=6.1901
step 10: loss=6.1071
step 11: loss=5.6148
step 12: loss=4.8965
step 13: loss=4.6399


## 第三步：把头换成 CNN

现在回答"如果我的头不是线性/门控 MLP 呢？" —— 换成沿特征轴的一维 CNN。
**骨架一行不动**：同样的状态声明位置、同样的 `predict`/`loss` 时机、同样的
groups 归并。只改字段形状和 `forward` 的数学。

一个要点：**参数按行就位**。`conv`/`mix`/`bias` 读到手时首轴就是打包行
（每行取自己 lineage 的版本），所以用逐行广播而不是 `torch.conv1d` 的全局卷积核。

In [3]:
CHANNELS = 16
KERNEL = 5


class CNNDistiller(pllm.Workload):
    """与 LinearDistiller 同骨架；头是深度可分离 CNN + 通道混合。"""

    head = pllm.RequestState(
        {"conv": (CHANNELS, KERNEL),
         "mix": (CHANNELS,),
         "bias": lambda m: (m.hidden_size,)},
        dtype=torch.float32,
        optimizer=pllm.AdamW(lr=3e-3),
        keys=(pllm.RequestKeyField.PROMPT_IDX, pllm.RequestKeyField.SAMPLE_IDX),
    )

    def forward(self, source, conv, mix, bias):
        pad = KERNEL // 2
        windows = torch.nn.functional.pad(source.float(), (pad, pad)).unfold(-1, KERNEL, 1)
        h = torch.nn.functional.silu((windows.unsqueeze(1) * conv.unsqueeze(2)).sum(-1))
        return source.float() + (h * mix.unsqueeze(-1)).sum(1) + bias

    def ready(self, versions: Annotated[torch.Tensor, pllm.StateVersion("head")]) -> torch.Tensor:
        return versions > 0

    @pllm.on(pllm.Phase.BEFORE_SAMPLE, rows="ready")
    @pllm.guide_sampling(beta=0.2)
    def predict(self, source: Annotated[torch.Tensor, pllm.Hidden(0)],
                conv: Annotated[torch.Tensor, pllm.Parameter("head", "conv")],
                mix: Annotated[torch.Tensor, pllm.Parameter("head", "mix")],
                bias: Annotated[torch.Tensor, pllm.Parameter("head", "bias")],
                versions: Annotated[torch.Tensor, pllm.StateVersion("head")]) -> torch.Tensor:
        return self.forward(source, conv, mix, bias)

    @pllm.on(pllm.Phase.AFTER_SAMPLE)
    @pllm.train(state="head", reduction="state_mean")
    def loss(self, source: Annotated[torch.Tensor, pllm.Hidden(0)],
             target: Annotated[torch.Tensor, pllm.Hidden(-1)],
             conv: Annotated[torch.Tensor, pllm.Parameter("head", "conv")],
             mix: Annotated[torch.Tensor, pllm.Parameter("head", "mix")],
             bias: Annotated[torch.Tensor, pllm.Parameter("head", "bias")],
             groups: Annotated[torch.Tensor, pllm.StateIndex("head")]) -> torch.Tensor:
        losses = (self.forward(source, conv, mix, bias) - target.float()).square().mean(1)
        counts = torch.bincount(groups).to(losses)
        return losses.new_zeros(counts.shape).index_add(0, groups, losses) / counts

In [4]:
import gc
llm.close()
del llm
gc.collect()
torch.cuda.empty_cache()

llm = pllm.make_llm(workload=CNNDistiller(), config=ModelConfig(
    "Qwen/Qwen2.5-0.5B-Instruct", VllmDType.BFLOAT16, 64, 2, 128, 0.2, seed=7))
try:
    outputs = llm.generate(["The sky is", "Water freezes at"],
        SamplingParams(max_tokens=8, temperature=0.0, ignore_eos=True), use_tqdm=False)
    for output in outputs:
        print(output.outputs[0].text)
    d = llm.worker_state().diagnostics
    print("losses:", d.loss_count, "updates:", d.update_count, "fallback:", d.fallback_count)
    assert d.loss_count > 0 and d.update_count > 0 and d.fallback_count == 0
    assert d.active_lineages == d.pending_state_releases == 0   # 请求结束，状态干净回收
finally:
    llm.close()

INFO 09-10 13:13:57 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 64, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 128, 'max_num_seqs': 2, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 13:13:57 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 13:13:58 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 13:13:58 [model.py:1883] Using max model len 64


INFO 09-10 13:13:58 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


INFO 09-10 13:14:00 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=64, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collec

... (9 outputs left out)

INFO 09-10 13:14:02 [kv_cache_utils.py:2236] Maximum concurrency for 64 tokens per request: 5073.25x


INFO 09-10 13:14:02 [gpu_worker.py:789] Free memory on device (11.63/23.48 GiB) on startup. Desired GPU memory utilization is (0.2, 4.7 GiB). Actual usage is 0.98 GiB for consumed memory (weights + non-torch), 0.0 GiB for peak activation, and 0.0 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=3832543232` (3.57 GiB) to fit into requested memory, or `--kv-cache-memory=11282413568` (10.51 GiB) to fully utilize gpu memory. Current kv cache memory in use is 3.72 GiB.


INFO 09-10 13:14:02 [core.py:355] init engine (profile, create kv cache, warmup model) took 0.91 s


 a beautiful place. It is a place
 0 degrees Celsius and boils at 
losses: 14 updates: 14 fallback: 0


## 要点回顾

- 训练 = 带优化器的 `RequestState` + 一个返回损失的 `@pllm.train` 函数；
  梯度、更新、版本发布由框架完成。
- 损失按行计算、按 `StateIndex` 归并到 lineage —— 多请求同批互不干扰。
- `ready` + `StateVersion` 是"还没训练好别生效"的标准门控。
- 换架构只动字段与 `forward`；时机、隔离、发布、回收不变。
- `active_lineages == 0`：请求结束，状态被精确回收。

下一篇：[教程 3：按请求路由的 LoRA](03_request_lora.ipynb)。